In [1]:
import sys
import os

# 프로젝트 루트 디렉토리 경로를 추가
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


In [2]:
import jax
import jax.numpy as jnp
import chex
import time
#disable jax JIT
#jax.config.update("jax_disable_jit", True)

from tqdm.autonotebook import trange, tqdm
from functools import partial
from JAxtar.bgpq import HashTableHeapValue, BGPQ

/tmp/ipykernel_1809420/1118636839.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import trange, tqdm


In [3]:
N = int(1e6)

In [4]:
sorted_a = jnp.sort(jax.random.uniform(jax.random.PRNGKey(0), (N,), minval=0, maxval=10))
sorted_b = jnp.sort(jax.random.uniform(jax.random.PRNGKey(1), (N,), minval=0, maxval=10))

2024-08-01 07:33:21.995897: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.5.82). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [5]:
@jax.jit
def merge_sorted(a, b):
    return jnp.sort(jnp.concatenate([a, b]))
sorted_c = merge_sorted(sorted_a, sorted_b)

time_start = time.time()
sorted_c = merge_sorted(sorted_a, sorted_b)
time_end = time.time()
print(f"JAX sort time: {time_end - time_start:.4f} sec")

JAX sort time: 0.0007 sec


In [6]:
def get_diagonal_indices(i, a_len, b_len, thread_len, a, b):
    index = i * (a_len + b_len) // thread_len
    a_top = jnp.where(index > a_len, a_len, index)
    b_top = jnp.where(index > a_len, index - a_len, 0)
    #print(a_top, b_top)
    a_bottom = b_top

    def _update(a_top, a_bottom, b_top, a_i, b_i):
        offset = (a_top - a_bottom) // 2
        a_i = a_top - offset
        b_i = b_top + offset

        cond1 = a[a_i] > b[b_i - 1]
        cond2 = a[a_i - 1] <= b[b_i]

        return a_i, b_i, cond1, cond2
    
    a_i, b_i, cond1, cond2 = _update(a_top, a_bottom, b_top, 0, 0)

    def _cond(val):
        a_i, b_i, a_top, b_top, a_bottom, cond1, cond2 = val
        return jnp.logical_not(jnp.logical_and(cond1, cond2))

    def _body(val):
        a_i, b_i, a_top, b_top, a_bottom, cond1, cond2 = val

        a_top, b_top = jax.lax.cond(
            jnp.logical_and(cond1, jnp.logical_not(cond2)),
            lambda _: (a_i - 1, b_i + 1),
            lambda _: (a_top, b_top),
            None
        )
        a_bottom = jnp.where(cond1, a_bottom, a_i + 1)

        a_i, b_i, cond1, cond2 = _update(a_top, a_bottom, b_top, a_i, b_i)

        return a_i, b_i, a_top, b_top, a_bottom, cond1, cond2
    
    a_i, b_i, a_top, b_top, a_bottom, cond1, cond2 = jax.lax.while_loop(
        _cond, _body, (a_i, b_i, a_top, b_top, a_bottom, cond1, cond2)
    )
    return a_i, b_i

@partial(jax.jit, static_argnums=(2,))
def parallel_merge(a, b, thread_len: int = 128):
    a_len = a.shape[0]
    b_len = b.shape[0]
    a_diag, b_diag = jax.vmap(get_diagonal_indices, in_axes=(0, None, None, None, None, None))(jnp.arange(thread_len-1)+1, a_len, b_len, thread_len, a, b)
    #merge
    a_indices = jnp.concatenate([jnp.array([0]), a_diag, jnp.array([a_len])])
    b_indices = jnp.concatenate([jnp.array([0]), b_diag, jnp.array([b_len])])
    return a_indices, b_indices

indices = parallel_merge(sorted_a, sorted_b, thread_len=1024)

time_start = time.time()
indices = parallel_merge(sorted_a, sorted_b, thread_len=1024)
time_end = time.time()
print(f"Parallel merge time: {time_end - time_start:.4f} sec")
#print(indices)


Parallel merge time: 0.0010 sec
